In [2]:
import pandas as pd
import numpy as np

# Using utf-8 this time - it handles Indian language characters correctly
df = pd.read_csv('../data/raw/zomato_india.csv', encoding='utf-8')

print("Shape:", df.shape)
print("\nSample names with special characters:")
print(df[df['name'].str.contains('é|í|á', na=False, regex= True)]['name'].head())

Shape: (211944, 26)

Sample names with special characters:
107     Pavilion Café - Jaypee Palace Hotel
865     Pavilion Café - Jaypee Palace Hotel
2773      MoMo Café - Courtyard By Marriott
3072                           Café Wrapico
3189      MoMo Café - Courtyard By Marriott
Name: name, dtype: object


In [3]:
# What establishment types exist?
print(df['establishment'].head(3).tolist())

# What does highlights actually contain?
print(df['highlights'].head(3).tolist())

# What's in delivery and takeaway?
print("\nDelivery values:", df['delivery'].value_counts())
print("\nTakeaway values:", df['takeaway'].value_counts())

# How many unique cites?
print(f"\nUnique cities: {df['city'].nunique()}")
print(df['city'].value_counts().head(10))


["['Quick Bites']", "['Quick Bites']", "['Quick Bites']"]
["['Lunch', 'Takeaway Available', 'Credit Card', 'Dinner', 'Cash', 'Air Conditioned', 'Indoor Seating', 'Pure Veg']", "['Delivery', 'No Alcohol Available', 'Dinner', 'Takeaway Available', 'Lunch', 'Cash', 'Indoor Seating']", "['No Alcohol Available', 'Dinner', 'Takeaway Available', 'Breakfast', 'Lunch', 'Cash', 'Delivery', 'Outdoor Seating', 'Air Conditioned', 'Self Service', 'Indoor Seating', 'Digital Payments Accepted', 'Pure Veg', 'Desserts and Bakes']"]

Delivery values: delivery
-1    132573
 1     78335
 0      1036
Name: count, dtype: int64

Takeaway values: takeaway
-1    211944
Name: count, dtype: int64

Unique cities: 99
city
Chennai      11630
Mumbai        6497
Bangalore     4971
Pune          4217
Lucknow       4121
Jabalpur      3994
New Delhi     3918
Jaipur        3713
Kochi         3370
Ajmer         3277
Name: count, dtype: int64


In [4]:
# Dropping coloumns we won't use
# - zipcode: mostly missing
# - url: Not usefull for analysis
# - country_id: all India anyway
# - currency: all 'Rs.' anyway

columns_to_drop = ['zipcode', 'url', 'country_id', 'currency']
df = df.drop(columns=columns_to_drop)

print(f"Shape after dropping columns: {df.shape}")

Shape after dropping columns: (211944, 22)


In [5]:
# Dropping coloumns we won't use
# - takeaway: too many '-1' values

df = df.drop(columns=['takeaway'])

In [6]:
import ast

# 'establishment' is a string that looks like "['Quick Bites']"
# We want to extract just "Quick Bites" as a clean string

def parse_establishment(value):
    """ Convert "['Quick Bites']" string into  'Quoick Bites'."""
    try:
        parsed = ast.literal_eval(value) # safely parses string to actual Python list
        if isinstance(parsed, list) and len(parsed) > 0:
            return parsed[0]
        return None
    except (ValueError, SyntaxError):
        return None
    
df['establishment'] = df['establishment'].apply(parse_establishment)

# Verify
print(df['establishment'].value_counts().head(10))


establishment
Quick Bites        64390
Casual Dining      61808
Café               22760
Bakery              8282
Dessert Parlour     7961
Bar                 6553
Fine Dining         6401
Sweet Shop          6103
Beverage Shop       5571
Dhaba               2939
Name: count, dtype: int64


In [7]:
df['delivery'] = df['delivery'].replace(-1, np.nan)
print(df['delivery'].isnull().sum())

132573


In [8]:
print(f"Before dedup: {len(df):,} rows")
df = df.drop_duplicates(subset=['res_id'], keep='first')
print(f"After dedup: {len(df):,} rows")

#df.to_csv('../data/processed/zomato_clean.csv', index=False, encoding='utf-8')

Before dedup: 211,944 rows
After dedup: 55,568 rows


In [9]:
# Saving the cleaned dataset

df.to_csv('../data/processed/zomato_clean.csv', index=False, encoding='utf-8')
print(f"Saved clean dataset: {df.shape}")

Saved clean dataset: (55568, 21)


In [10]:
# Top 10 cities by restaurant count
print(df['city'].value_counts().head(10))

# Top 10 establishment types
print(df['establishment'].value_counts().head(10))

# Average rating overall
print(f"\nAverage rating across India: {df['aggregate_rating'].mean():.2f}")


city
Bangalore    2247
Mumbai       2022
Pune         1843
Chennai      1827
New Delhi    1704
Jaipur       1395
Kolkata      1361
Ahmedabad    1247
Goa          1150
Lucknow      1135
Name: count, dtype: int64
establishment
Quick Bites        14032
Casual Dining      12270
Café                4123
Bakery              3741
Dessert Parlour     3675
Sweet Shop          2615
Beverage Shop       2440
Fine Dining         1535
Food Court          1494
Bar                 1399
Name: count, dtype: int64

Average rating across India: 2.96


In [11]:
zero_rated = (df['aggregate_rating'] == 0).sum()
print(f"Zero-rated restaurants: {zero_rated} ({zero_rated/len(df)*100:.1f}%)")

# Average rating exclusive zeros
real_avg = df[df['aggregate_rating'] > 0]['aggregate_rating'].mean()
print(f"Average rating (excluding zeros): {real_avg:.2f}")

print(df['aggregate_rating'].describe())

Zero-rated restaurants: 10058 (18.1%)
Average rating (excluding zeros): 3.61
count    55568.000000
mean         2.958593
std          1.464576
min          0.000000
25%          2.900000
50%          3.500000
75%          3.900000
max          4.900000
Name: aggregate_rating, dtype: float64
